# Gold Mart Creation and Validation

This notebook creates and validates the final Gold dimensional model for the NYC Green Taxi project.

The model follows a star schema designed for trip-level analysis and business reporting.

%md
## Purpose

The purpose of this notebook is to transform the trusted Silver-layer data into reusable Gold dimensions and a trip-level fact table.

The final model will support analysis of trip dates, trip times, taxi zones, weather conditions, revenue, distance, duration, and passenger behavior.

%md
## Approved Gold Tables

The final Gold layer will contain the following tables:

- `dim_date`
- `dim_time`
- `dim_taxi_zone`
- `dim_weather_hour`
- `fact_green_taxi_trip`

The fact table grain is one row per Green Taxi trip.

%md
## Silver Source Inspection

The Gold layer will be created from the following Silver source tables:

- `green_taxi`
- `taxi_zones`
- `weather`

Before creating the Gold tables, we inspect the source structures and sample records to confirm the available columns and data types.

In [0]:
SHOW TABLES IN `ftw-week-08`.`02_silver`;

In [0]:
DESCRIBE TABLE `ftw-week-08`.`02_silver`.green_taxi;

In [0]:
SELECT *
FROM `ftw-week-08`.`02_silver`.green_taxi
LIMIT 10;

## Green Taxi Table Structure

The `green_taxi` Silver table contains the trip-level source data.

It provides the pickup and dropoff timestamps, pickup and dropoff location identifiers, trip distance, trip duration, passenger count, and financial measures such as fare, tip, tolls, and total amount.

This table will become the main source for `fact_green_taxi_trip`.


%md
## Green Taxi Column Inventory

The Green Taxi columns were reviewed to identify the fields needed for the fact table.

The important fields include the trip timestamps, location identifiers, trip measures, financial measures, and `batch_id` for lineage.

The source does not provide a guaranteed unique trip identifier, so a deterministic `trip_key` will be generated from stable trip attributes.

In [0]:
DESCRIBE TABLE `ftw-week-08`.`02_silver`.taxi_zones;

In [0]:
SELECT *
FROM `ftw-week-08`.`02_silver`.taxi_zones
LIMIT 10;

## Taxi Zone Table Structure

The `taxi_zones` table provides the descriptive information for each taxi location identifier.

This table will be transformed into `dim_taxi_zone`, which will be used for both pickup-zone and dropoff-zone analysis.

In [0]:
DESCRIBE TABLE `ftw-week-08`.`02_silver`.weather;

In [0]:
SELECT *
FROM `ftw-week-08`.`02_silver`.weather
ORDER BY weather_datetime
LIMIT 10;

## Weather Column Inventory and Preview

The `weather` table contains hourly weather observations identified by `weather_datetime`.

The available weather attributes include temperature, precipitation, rain, snowfall, weather code, and wind speed.

The weather dimension will be joined to trips using the pickup time rounded to the matching weather hour.


## Final Gold Dimensional Model

The approved star schema contains four dimensions and one fact table.

The fact table stores one row per Green Taxi trip. The dimensions provide reusable descriptive attributes for dates, times, taxi zones, and hourly weather.

Pickup and dropoff dates, times, and zones will be treated as role-playing dimensions in the fact table.

%md
## Existing Gold Table Assessment

The existing Gold schema contains tables from the previous setup design:

- `dim_datetime`
- `dim_location`
- `dim_weather`
- `fact_taxi_trips`

These tables are retained for reference and will not be used in the final implementation because their design does not match the newly approved star schema.

The final implementation will use the new tables:

- `dim_date`
- `dim_time`
- `dim_taxi_zone`
- `dim_weather_hour`
- `fact_green_taxi_trip`

In [0]:
CREATE OR REPLACE TABLE `ftw-week-08`.`03_gold`.dim_date
USING DELTA
AS
WITH date_range AS (
    SELECT
        MIN(TO_DATE(lpep_pickup_datetime)) AS min_date,
        MAX(TO_DATE(lpep_pickup_datetime)) AS max_date
    FROM `ftw-week-08`.`02_silver`.green_taxi
),
dates AS (
    SELECT EXPLODE(
        SEQUENCE(min_date, max_date, INTERVAL 1 DAY)
    ) AS full_date
    FROM date_range
)
SELECT
    CAST(DATE_FORMAT(full_date, 'yyyyMMdd') AS INT) AS date_key,
    full_date,
    YEAR(full_date) AS year,
    QUARTER(full_date) AS quarter,
    MONTH(full_date) AS month_number,
    DATE_FORMAT(full_date, 'MMMM') AS month_name,
    DAYOFMONTH(full_date) AS day_of_month,
    DATE_FORMAT(full_date, 'EEEE') AS day_name,
    DAYOFWEEK(full_date) AS day_of_week,
    CASE
        WHEN DAYOFWEEK(full_date) IN (1, 7) THEN TRUE
        ELSE FALSE
    END AS is_weekend,
    FALSE AS is_holiday
FROM dates;

## Create DIM_DATE

The `dim_date` table was created from the minimum and maximum Green Taxi pickup dates.

A complete calendar row was generated for every date in that range. Each date has a unique `date_key` and descriptive calendar attributes such as year, quarter, month, day, weekend indicator, and holiday indicator.

In [0]:
SELECT
    COUNT(*) AS row_count,
    MIN(full_date) AS minimum_date,
    MAX(full_date) AS maximum_date,
    COUNT(DISTINCT date_key) AS unique_date_keys
FROM `ftw-week-08`.`03_gold`.dim_date;

In [0]:
SELECT *
FROM `ftw-week-08`.`03_gold`.dim_date
ORDER BY full_date
LIMIT 10;

In [0]:

DESCRIBE TABLE `ftw-week-08`.`03_gold`.dim_date;

%md
## DIM_DATE Validation Result

The `dim_date` table was successfully created and validated.

The table contains 6,362 rows and 6,362 unique date keys. The date range covers 2008-12-31 to 2026-06-01.

The preview and schema inspection confirmed that the expected date attributes and data types are present. The dimension is ready to support the Gold fact table.

## Create DIM_TIME

The `dim_time` table contains one row for every hour of the day.

It provides reusable time attributes for pickup-hour and dropoff-hour analysis. Its expected grain is one row per hour.

In [0]:
CREATE OR REPLACE TABLE `ftw-week-08`.`03_gold`.dim_time
USING DELTA
AS
SELECT
    hour_number AS time_key,
    hour_number,
    CONCAT(
        LPAD(CAST(hour_number AS STRING), 2, '0'),
        ':00'
    ) AS time_label,
    CASE
        WHEN hour_number = 0 THEN 12
        WHEN hour_number <= 12 THEN hour_number
        ELSE hour_number - 12
    END AS hour_12,
    CASE
        WHEN hour_number < 12 THEN 'AM'
        ELSE 'PM'
    END AS am_pm,
    CURRENT_TIMESTAMP() AS gold_created_at
FROM (
    SELECT EXPLODE(SEQUENCE(0, 23)) AS hour_number
);

In [0]:
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT time_key) AS unique_time_keys,
    MIN(time_key) AS minimum_time_key,
    MAX(time_key) AS maximum_time_key
FROM `ftw-week-08`.`03_gold`.dim_time;